In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/vit-imagerecognition-outputs/kaggle_output/vit-base-patch16-224-in21k/.locks/models--google--vit-base-patch16-224-in21k/70fbc148eb26a06bac351d46fddc0a23037b4ce4.lock
/kaggle/input/vit-imagerecognition-outputs/kaggle_output/vit-base-patch16-224-in21k/.locks/models--google--vit-base-patch16-224-in21k/fd4e1169c7aa6c2dbfa8a6448be13b35abc0ee256190857c90009d12c094619b.lock
/kaggle/input/vit-imagerecognition-outputs/kaggle_output/vit-base-patch16-224-in21k/.locks/models--google--vit-base-patch16-224-in21k/254071bbf72dd0fd535b61768d9cd87adcb776f4.lock
/kaggle/input/vit-imagerecognition-outputs/kaggle_output/vit-base-patch16-224-in21k/models--google--vit-base-patch16-224-in21k/refs/main
/kaggle/input/vit-imagerecognition-outputs/kaggle_output/vit-base-patch16-224-in21k/models--google--vit-base-patch16-224-in21k/blobs/70fbc148eb26a06bac351d46fddc0a23037b4ce4
/kaggle/input/vit-imagerecognition-outputs/kaggle_output/vit-base-patch16-224-in21k/models--google--vit-base-patch16-224-in21

In [3]:
! pip install lightning mlflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 818.9/818.9 kB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 28.2/28.2 MB 66.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 101.9 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 681.0/681.0 kB 35.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.9/94.9 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.4/203.4 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 4.8 MB/s eta 0:00:00


In [4]:
import numpy as np
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
import h5py
import lightning as L
import torch.nn as nn
import torch.optim as optim
import numpy as np
import sys
import torchvision.transforms as transforms
from PIL import Image
import random
import matplotlib.pyplot as plt
import logging
from sklearn.model_selection import train_test_split
import mlflow
from torch.utils.data import Dataset
import torch
import torchvision.transforms as transforms
from PIL import Image
import random

from transformers import ViTImageProcessor, ViTModel, ViTConfig
from transformers.utils import cached_file

from lightning.pytorch import Trainer
from lightning.pytorch.loggers import MLFlowLogger
from datetime import datetime

import torch.nn.functional as F
from lightning.pytorch.callbacks import Callback, ModelCheckpoint, EarlyStopping

In [12]:
# enable GPU
if torch.cuda.is_available():
    print("✅ GPU is available!")
    print("Device name:", torch.cuda.get_device_name(0))
else:
    print("❌ GPU not available. Using CPU.")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

✅ GPU is available!
Device name: Tesla P100-PCIE-16GB
Using device: cuda


In [19]:
# add kaggle path
kaggle_output="/kaggle/working/"

os.makedirs("/kaggle/working/vit-base-patch16-224-in21k", exist_ok=True)

In [20]:
# load pre defined parameters
local_config_path = "/kaggle/working/vit-base-patch16-224-in21k"

# label encoder
label_encoder = {
    "dot": 0,
    "scatter": 1,
    "horizontal_bar": 2,
    "line": 3,
    "vertical_bar": 4,
}

# label decoder
label_decoder = {v: k for k, v in label_encoder.items()}

id2label = {idx: label for idx, label in label_decoder.items()}
label2id = {label: idx for idx, label in label_encoder.items()}

# Dataset parameters
image_cls_height = 320
image_cls_width = 320

# dataset size multiplier
ds_multiplier = 2

# Datamodule Parameters
initial_augment_prob = 0.8
final_augment_prob = 0.2

# Dataloader parameters
num_workers = 4
batch_size = 32
num_epochs = 30

# data path
# hdf5_file = "/kaggle/input/benetech-ds/ImageTransformation.h5"

timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")

In [21]:
mlflow_logger = MLFlowLogger(
    experiment_name="ViT-image-recognition",
    tracking_uri="/kaggle/working/mlruns"
)

In [22]:
# load VIT model's config and processor

config = ViTConfig.from_pretrained("google/vit-base-patch16-224-in21k",cache_dir=local_config_path)
processor = ViTImageProcessor.from_pretrained('google/vit-base-patch16-224-in21k', cache_dir=local_config_path)

config.json:   0%|          | 0.00/502 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

In [23]:
# add customized paramaters into config file

config.num_labels = len(id2label)
config.id2label = id2label
config.label2id = label2id
config.hidden_dropout_prob = 0.1 
config.image_size = 320
config.patch_size = 16
# 320/16=20,
# input token num: 320*320 / 16*16 = 20*20=400 
# add CLS: 400+1=401
# each token seq len: 16*16*3=768

model_name = "google/vit-base-patch16-224-in21k"
config_path = cached_file(model_name, "config.json")

config.json:   0%|          | 0.00/502 [00:00<?, ?B/s]

In [24]:
# load pretrained VIT model

model = ViTModel.from_pretrained(
    "google/vit-base-patch16-224-in21k",
    config=config,
    # I want fine tuning the model with Higher Resolution, change the input size from 224 to 320
    # so set ignore mismatched_size right here
    ignore_mismatched_sizes=True,
    cache_dir=local_config_path,
)

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Some weights of ViTModel were not initialized from the model checkpoint at google/vit-base-patch16-224-in21k and are newly initialized because the shapes did not match:
- embeddings.position_embeddings: found shape torch.Size([1, 197, 768]) in the checkpoint and torch.Size([1, 401, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [27]:
class ViTClassifier(L.LightningModule):
    def __init__(self, pretrained_model, config, num_class, lr=None):
        """
        Params:
            pretrained_model: The loaded ViT model
            config:(transformers.models.vit.configuration_vit.ViTConfig) the model's config file
            num_class: the numer of classes
            lr: learning rate
        """
        super().__init__()
        # saving the information in checkpoints and YAML files
        self.save_hyperparameters()

        self.model = pretrained_model
        self.config = config
        self.num_class = num_class
        self.learning_rate = (
            lr if lr is not None else self.hparams.get("learning_rate", 1e-4)
        )

        # define what layers in the pretrained model should be fine-tuning
        # fine tune the embedding (positional embedding) layer
        for param in self.model.embeddings.parameters():
            param.requires_grad = True

        # freeze the first 9 layers(total 12)
        for name, param in self.model.encoder.named_parameters():
            if any(f"layer.{i}" in name for i in range(9)):
                param.requires_grad = False

        # fine tuning the last 3 layers
        for name, param in self.model.encoder.named_parameters():
            if any(f"layer.{i}" in name for i in range(9, 12)):
                param.requires_grad = True

        # fine tune pooling layer
        for param in self.model.pooler.parameters():
            param.requires_grad = True

        # define the classfier head
        self.classifier = nn.Linear(
            in_features=config.hidden_size, out_features=self.num_class
        )

    def forward(self, x):
        outputs = self.model(x)
        # In VIT model, we rely on CLS token to do classification
        cls_token = outputs.last_hidden_state[:, 0, :]
        # classifier head
        logits = self.classifier(cls_token)

        return logits

    def training_step(self, batch, batch_idx):
        images, labels = batch
        images = images.to(self.device)
        labels = labels.to(self.device)

        logits = self(images)
        loss = F.cross_entropy(logits, labels)
        self.log("train_loss", loss, prog_bar=True, on_step=False, on_epoch=True)

        return loss

    def validation_step(self, batch, batch_idx):
        images, labels = batch
        images = images.to(self.device)
        labels = labels.to(self.device)
        logits = self(images)
        val_loss = F.cross_entropy(logits, labels)
        acc = (logits.argmax(dim=1) == labels).float().mean()
        self.log("val_loss", val_loss, prog_bar=True, on_step=False, on_epoch=True)
        self.log("val_acc", acc, prog_bar=True, on_step=False, on_epoch=True)

        return {"val_loss": val_loss, "val_acc": acc}

    def predict_step(self, batch, batch_idx):
        images, _ = batch
        logits = self(images)
        preds = torch.argmax(logits, dim=1)

        return preds

    def on_train_epoch_end(self):

        train_loss = self.trainer.callback_metrics.get("train_loss")
        if train_loss is not None:
            print(f"Train Epoch {self.current_epoch + 1} - Avg Loss: {train_loss:.4f}")

        # mlflow
        self.logger.log_metrics(
            {
                "manual_record_train_loss": train_loss.item()
            },
            step=self.current_epoch,
        )

    def on_validation_epoch_end(self):

        val_loss = self.trainer.callback_metrics.get("val_loss")
        val_acc = self.trainer.callback_metrics.get("val_acc")
        if val_loss is not None and val_acc is not None:
            print(
                f"Val Epoch {self.current_epoch + 1} - Loss: {val_loss:.4f}, Acc: {val_acc:.4f}"
            )

        # mlflow
        self.logger.log_metrics(
            {
                "manual_record_val_loss": val_loss.item(),
                "manual_record_val_acc": val_acc.item(),
            },
            step=self.current_epoch,
        )

    def on_fit_start(self):
        print(f"✅ Model is on device: {next(self.parameters()).device}")
        if self.logger:
            self.logger.log_hyperparams(self.hparams)

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.parameters(), lr=self.learning_rate)

        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode="min",
            factor=0.5,
            patience=2, # trigger lr scheduler if without imporvement 3 times
            verbose=True,
        )

        # decrease lr based off val loss automatically
        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "monitor": "val_loss",
                "interval": "epoch",
                "frequency": 1, # do step each 1 epoch
            },
        }

In [28]:
ckpt_path = "/kaggle/input/vit-imagerecognition-outputs/kaggle_output/ViTCheckpoints/20250322-143509/vit-best-epoch=12-val_acc=0.9985.ckpt"
model = ViTClassifier.load_from_checkpoint(ckpt_path)

/usr/local/lib/python3.10/dist-packages/lightning/pytorch/utilities/parsing.py:209: Attribute 'pretrained_model' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['pretrained_model'])`.


In [29]:
torch.save(model.state_dict(), "/kaggle/working/vit-best-epoch=12-val_acc=0.9985.pt")